<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

This installs the core scientific and visualization libraries needed for the notebook.

In [ ]:
!pip install scipy numpy matplotlib

The TA-Lib package requires special installation steps. TA-Lib depends on a C library that must be installed at the system level before the Python wrapper will work (see ta-lib.org for platform-specific instructions). Install separately before running this notebook.

## Imports and setup

We use math for the constant pi, scipy for the Butterworth bandpass filter functions (butter and filtfilt), numpy for logarithmic and array operations, talib for Hilbert Transform cycle detection indicators, matplotlib for plotting, and openbb for loading forex market data.

In [ ]:
from math import pi
import matplotlib.pyplot as plt
import numpy as np
import talib
import yfinance as yf
from scipy.signal import butter, filtfilt

## Load and prepare EUR/USD data

We load six years of EUR/USD daily data to give the filter enough history to detect meaningful cycles across different market regimes.

In [ ]:
data = yf.download(
    "EURUSD=X",
    start="2016-01-01",
    end="2021-12-31",
    multi_level_index=False
)

We extract the close column into its own DataFrame and compute log returns, which are the standard way to measure price changes in quantitative finance because they're additive over time.

In [ ]:
prices = (
    data.Close
    .to_frame()
    .rename(
        columns={
            "Close": "close"
        }
    )
)

In [ ]:
prices["log_return"] = (
    prices.close
    .apply(np.log)
    .diff(1)
)

Log returns normalize price movements so that a 1% move up and a 1% move down are symmetric. This matters when we later measure the amplitude of filtered cycles, because raw price differences would skew our strength readings depending on the price level at the time.

## Detect cycles and build the bandpass filter

The Hilbert Transform estimates the dominant cycle's phase and period directly from price data. We convert the phase into a sine wave that oscillates between -1 and +1, giving us a clean trading signal that peaks and troughs with the detected cycle.

In [ ]:
prices["phase"] = talib.HT_DCPHASE(prices.close)

In [ ]:
prices["signal"] = np.sin(prices.phase + pi / 4)

In [ ]:
prices["period"] = talib.HT_DCPERIOD(prices.close)

The phase shift of pi/4 (45 degrees) nudges the signal forward slightly so that it aligns better with actionable turning points rather than lagging behind them. The dominant cycle period tells the bandpass filter exactly which frequency band to isolate, so the filter adapts as market conditions change rather than using a fixed window.

This function builds a second-order Butterworth bandpass filter centered on the detected cycle period. The delta parameter controls how wide the frequency band is around that center, and filtfilt applies the filter forward and backward to avoid introducing any time lag.

In [ ]:
def butter_bandpass(data, period, delta=0.5, fs=5):
    nyq = 0.5 * fs

    low = 1.0 / (period * (1 + delta))
    low /= nyq

    high = 1.0 / (period * (1 - delta))
    high /= nyq

    b, a = butter(2, [low, high], btype="band")

    return filtfilt(b, a, data)

The Butterworth design produces the flattest possible response within the passband, meaning it treats all frequencies inside the band equally rather than favoring some over others. Using filtfilt (zero-phase filtering) is critical here because a standard one-pass filter would shift the signal in time, making our cycle peaks appear earlier or later than they actually occurred. That kind of time shift would create false entry and exit points.

This helper function applies the bandpass filter on a rolling 30-day window, using the most recent dominant cycle period to tune the filter for each window. Rolling application means the filter adapts over time instead of using one static setting for the entire dataset.

In [ ]:
def roll_apply(e):
    close = prices.close.loc[e.index]
    period = prices.period.loc[e.index][-1]
    out = butter_bandpass(close, period)
    return out[-1]

In [ ]:
prices["filtered"] = (
    prices
    .dropna()
    .rolling(window=30)
    .apply(
        lambda series: roll_apply(series), raw=False
    )
    .iloc[:, 0]
)

We pass raw=False so that pandas sends each rolling window as a Series with its original index intact. This lets roll_apply look up the matching close prices and period values by date. Without this, we'd lose the index alignment and the filter would read the wrong data.

## Measure cycle strength and generate positions

We measure the amplitude of the filtered signal over a 30-day rolling window, then smooth it with an exponential moving average. Amplitude tells us how strong the detected cycle is right now, which is the key to deciding whether the pattern is worth trading.

In [ ]:
prices["amplitude"] = (
    prices
    .filtered
    .rolling(window=30)
    .apply(
        lambda series: series.max() - series.min()
    )
)

In [ ]:
prices["ema_amplitude"] = (
    talib
    .EMA(
        prices.amplitude,
        timeperiod=30,
    )
)

This is the core idea from the post: professionals don't trade every cycle they detect. They measure whether the cycle is strong enough to act on. A weak amplitude means the cycle is barely present in the data, and trading it would be reacting to noise. The EMA smoothing prevents us from flipping in and out of trades based on one or two outlier days.

We combine the cycle signal and amplitude into trading rules. A position is taken only when the signal crosses a threshold AND the cycle amplitude exceeds 40 pips, ensuring we trade only when the pattern is both directionally clear and strong enough to profit from.

In [ ]:
signal_thresh = 0.75
amp_thresh = 0.004  # 40 pips

In [ ]:
prices["position"] = 0
prices.loc[
    (prices.signal >= signal_thresh)
    & (prices.amplitude > amp_thresh), "position"
] = -1
prices.loc[
    (prices.signal <= -signal_thresh)
    & (prices.amplitude > amp_thresh), "position"
] = 1

When the sine signal is near +1, the cycle is at its peak and we go short (expecting a move down). When it's near -1, the cycle is at its trough and we go long. The amplitude gate is what makes this a filter-based strategy rather than a naive oscillator strategy. Without it, we'd be entering trades during flat, directionless markets where the cycle has no real predictive power.

We plot the smoothed amplitude with its threshold, the cycle signal with its entry thresholds, and the resulting position series. This three-panel view lets us visually confirm that trades only occur when both conditions are met simultaneously.

In [ ]:
fig, axes = plt.subplots(
    nrows=3,
    figsize=(15, 10),
    sharex=True,
)
prices.ema_amplitude.plot(
    ax=axes[0],
    title="amp",
)
axes[0].axhline(
    amp_thresh,
    lw=1,
    c="r",
)
prices.signal.plot(
    ax=axes[1],
    title="signal",
)
axes[1].axhline(
    signal_thresh,
    c="r",
)
axes[1].axhline(
    -signal_thresh,
    c="r",
)
prices.position.plot(
    ax=axes[2],
    title="position",
)
fig.tight_layout()
plt.show()

The red horizontal lines are our decision boundaries. In the amplitude panel, trades are only allowed when the line is above the red threshold. In the signal panel, entries happen only when the sine wave pushes past the upper or lower red lines. The position panel should show long stretches of zero (no trade) interrupted by short bursts of +1 or -1, which is exactly the selective, filter-first behavior the post describes.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.